# 02 — Data Cleaning & Panel Construction

Runs the `clean.py` pipeline end-to-end: loads each raw source, cleans and standardises
country codes, merges all sources into a single panel, constructs derived variables
(NEP, GDP share, lags), and validates coverage for focal countries.
Output: `data/processed/panel_model_ready.csv`.

In [ ]:
import sys
sys.path.append('../src')
from clean import run_full_pipeline, clean_owid_energy, clean_world_bank, clean_cofer, clean_chinn_ito, clean_governance, merge_panel
from variables import construct_thorium_monetary_potential, flag_focal_countries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

PROCESSED = '../data/processed/'
RAW = '../data/raw/'

## 1. Run Full Pipeline

In [ ]:
panel = run_full_pipeline()
print(f"\nPanel shape: {panel.shape}")
panel.head()

## 2. Coverage Diagnostics

In [ ]:
focal_iso = ['CHN', 'IND', 'RUS', 'JPN', 'USA', 'GBR', 'DEU', 'CHE', 'AUS', 'CAN', 'BRA']
key_vars = ['net_energy_position', 'gdp_share', 'trade_openness',
            'inflation_cpi', 'reserve_share', 'kaopen',
            'net_energy_position_lag10', 'net_energy_position_lag15']

# Build coverage matrix
focal_panel = panel[panel['country_code'].isin(focal_iso)]
available_vars = [v for v in key_vars if v in focal_panel.columns]
coverage = focal_panel.groupby('country_code')[available_vars].apply(
    lambda x: x.notna().mean()
).reindex([iso for iso in focal_iso if iso in focal_panel['country_code'].values])

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(coverage, annot=True, fmt='.0%', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Variable Coverage by Country\n(% non-null country-years)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 3. Reserve Currency Entities

In [ ]:
reserve_panel = panel[panel['has_reserve_share'] == True].copy()
print(f"Reserve currency country-years: {len(reserve_panel)}")
print(f"Entities: {sorted(reserve_panel['country_code'].unique())}")
print(f"Year range: {reserve_panel['year'].min()}\u2013{reserve_panel['year'].max()}")
print()

# Summary stats for the model variables
model_vars = ['reserve_share', 'net_energy_position', 'gdp_share',
              'trade_openness', 'inflation_cpi']
available = [v for v in model_vars if v in reserve_panel.columns]
reserve_panel[available].describe().round(4)

## 4. Focal Country Year Coverage

In [ ]:
focal_full = panel[panel['country_code'].isin(['CHN', 'IND', 'RUS', 'JPN'])].copy()
focal_full['decade'] = (focal_full['year'] // 10) * 10

# Count non-null NEP observations per country per decade
if 'net_energy_position' in focal_full.columns:
    coverage_decade = focal_full.groupby(['country_code', 'decade'])['net_energy_position'].count().unstack(fill_value=0)
    coverage_decade.plot(kind='bar', figsize=(10, 4), colormap='tab10')
    plt.title('NEP Observations per Decade \u2014 Focal Countries')
    plt.ylabel('N observations')
    plt.xlabel('Country')
    plt.xticks(rotation=0)
    plt.legend(title='Decade', bbox_to_anchor=(1.02, 1))
    plt.tight_layout()
    plt.show()

## 5. Save Panel

In [ ]:
import os
os.makedirs(PROCESSED, exist_ok=True)
out_path = f'{PROCESSED}panel_model_ready.csv'
panel.to_csv(out_path, index=False)
print(f"Panel saved: {out_path}")
print(f"Shape: {panel.shape}")
print(f"\nColumns ({len(panel.columns)}):")
for col in sorted(panel.columns):
    n_null = panel[col].isna().sum()
    print(f"  {col:<45} {n_null:>6} nulls ({100*n_null/len(panel):.1f}%)")